# Gaussianity checks for `mi_importance.py`

[mi_importance.py](mi_importance.py) scores channels with a closed-form Gaussian conditional MI:

    I(S ; T | rest) = 1/2 * log( det Omega_full[S,S] / det Omega_base[S,S] )

That formula is exact only if `[X, cond, T]` is jointly Gaussian. This notebook captures the
activations the estimator actually sees and checks how far they are from that.

| test | question |
|---|---|
| 1. Marginals | Are single channels bell-shaped? |
| 2. Joint shape | Is the *joint* distribution Gaussian, not just the marginals? |
| 3. Gaussian vs KSG MI | Does assuming Gaussianity change the channel ranking? |
| 4. Conditioning on `t` | Does conditioning on `t` remove `t`'s effect, or only its mean? |
| 5. Stability | Is the score stable across shrinkage and data splits? |

Tests 3 and 4 are the ones that can change a pruning decision.

## Setup

In [ ]:
!git clone --branch mipp-lookahead2 https://github.com/elliotcanter11/Diff-Pruning.git

In [ ]:
%cd Diff-Pruning/

In [ ]:
!pip install -r requirements.txt

In [ ]:
!python tools/extract_cifar10_hug.py --output data

In [ ]:
!bash tools/convert_cifar10_ddpm_ema.sh

## Capture

Runs the calibration loop from [ddpm_prune.py](ddpm_prune.py) (`mi_calibrate`) and keeps the buffers
`MIImportance` fills, so the tests below see exactly what the estimator sees:

* `imp._loc_buf[conv]` — `(n_images*LOCS, C)`, one row per **(image, sampled pixel)**. Used by the
  adjacency and mid-range terms.
* `imp._img_buf[conv]` — `(n_images, C*g*g)`, one row per **image**, each channel spatially pooled.
  Used by the output term.
* `imp._loc_cond` — `(rows, 15)`: 9 Fourier features of `t`, then 6 position features.

In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.auto import tqdm

# compat shims, copied from ddpm_prune.py (diffusers/ here is the vendored copy)
import huggingface_hub
from huggingface_hub import constants as hf_constants
if not hasattr(hf_constants, "hf_cache_home"):
    hf_constants.hf_cache_home = hf_constants.HF_HUB_CACHE
if not hasattr(huggingface_hub, "cached_download"):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download
if not hasattr(huggingface_hub, "HfFolder"):
    class HfFolder:
        @staticmethod
        def get_token(): return huggingface_hub.get_token()
    huggingface_hub.HfFolder = HfFolder
import jax
if not hasattr(jax.random, "KeyArray"): jax.random.KeyArray = jax.Array
import transformers.utils as tf_utils
if not hasattr(tf_utils, "FLAX_WEIGHTS_NAME"): tf_utils.FLAX_WEIGHTS_NAME = "flax_model.msgpack"

from diffusers import DDPMPipeline
from mi_importance import MIImportance
from torchvision import transforms as T
import utils

DEVICE  = 'cuda:0'
BATCH   = 128
BATCHES = 8      # enough for tests 1-4; raise for test 5
LOCS    = 4      # = --mi_num_locations
OUT_G   = 1      # = --mi_out_grid

pipeline = DDPMPipeline.from_pretrained('pretrained/ddpm_ema_cifar10').to(DEVICE)
model, scheduler = pipeline.unet.eval(), pipeline.scheduler

tf = T.Compose([T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean=0.5, std=0.5)])
loader = torch.utils.data.DataLoader(
    utils.get_dataset('data/cifar10_images', transform=tf),
    batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)

imp = MIImportance(num_locations=LOCS, out_grid=OUT_G).attach(model, ignored_layers=[model.conv_out])

t_raw, it = [], iter(loader)
with torch.no_grad():
    for _ in tqdm(range(BATCHES), desc='calibrating'):
        batch = next(it)
        batch = batch[0] if isinstance(batch, (list, tuple)) else batch
        batch = batch.to(DEVICE)
        t = torch.randint(0, scheduler.config.num_train_timesteps, (batch.shape[0],), device=DEVICE).long()
        noisy = scheduler.add_noise(batch, torch.randn_like(batch), t)
        imp.new_pass(batch.shape[0])
        out = model(noisy, t).sample
        imp.record_output(out)
        imp.record_timesteps(t)
        t_raw.append(t.cpu())
imp.finalize()

t_img  = torch.cat(t_raw).numpy()                    # (n_images,)
t_loc  = np.repeat(t_img, LOCS)                      # (rows,), aligned with _loc_buf
coords = torch.cat(imp._coords_buf).numpy()          # (rows, 2), normalised (u, v)
COND   = imp._loc_cond.numpy()                       # (rows, 15) = [t features | position features]
COND_T, COND_P = COND[:, :9], COND[:, 9:]

convs = [m for m, _ in sorted(imp._order.items(), key=lambda kv: kv[1]) if imp._loc_buf.get(m) is not None]
name_of = {m: n for n, m in model.named_modules()}
PICKS = [convs[i] for i in sorted({0, len(convs)//4, len(convs)//2, 3*len(convs)//4, len(convs)-1})]

loc = lambda m: imp._loc_buf[m].float().numpy()      # (rows, C)
img = lambda m: imp._img_buf[m].float().numpy()      # (n_images, C*g*g)

print(f'{len(convs)} convs captured | loc rows = {loc(convs[0]).shape[0]} | images = {len(t_img)}')
for m in PICKS:
    print(f'  [{imp._order[m]:>2}] {name_of[m]:<55} C={loc(m).shape[1]}')

## 1. Marginals

Per-channel skew and excess kurtosis (0 = Gaussian), for both sample types.

Gaussian marginals are necessary but not sufficient for the joint Gaussianity the formula needs, so
this is a first look rather than a verdict. What to look for:

* `kurt med` large — heavy tails, so a few outlier pixels set the covariance.
* `kurt IQR` large — channels differ a lot from each other. Non-Gaussianity that hits every channel
  equally mostly cancels out of a ranking; uneven non-Gaussianity is what mis-ranks channels.
* `pooled` much tamer than `per-loc` — expected, since pooled rows are averages over H*W. It is a
  reason to trust the output term more than the adjacency and mid-range terms.

In [ ]:
def moments(X, max_ch=96, max_rows=6000, seed=0):
    r = np.random.default_rng(seed)
    cols = r.choice(X.shape[1], min(max_ch, X.shape[1]), replace=False)
    rows = r.choice(X.shape[0], min(max_rows, X.shape[0]), replace=False)
    Z = X[np.ix_(rows, cols)]
    return stats.skew(Z, 0), stats.kurtosis(Z, 0)

hdr = (f"{'layer':<38}{'samples':<10}{'|skew|':>9}{'kurt med':>10}"
       f"{'kurt p90':>10}{'kurt max':>10}{'kurt IQR':>10}")
print(hdr); print('-' * len(hdr))
for m in PICKS:
    for kind, X in (('per-loc', loc(m)), ('pooled', img(m))):
        sk, ku = moments(X)
        print(f'{name_of[m][-37:]:<38}{kind:<10}{np.median(abs(sk)):>9.2f}{np.median(ku):>10.2f}'
              f'{np.percentile(ku, 90):>10.2f}{ku.max():>10.1f}'
              f'{np.subtract(*np.percentile(ku, [75, 25])):>10.2f}')

In [ ]:
# histogram (log y) + QQ plot, for a typical and the heaviest-tailed channel at three depths
show = [PICKS[0], PICKS[len(PICKS)//2], PICKS[-1]]
fig, ax = plt.subplots(len(show), 4, figsize=(15, 3.1*len(show)))
for i, m in enumerate(show):
    X = loc(m)
    _, ku = moments(X)
    cols = np.random.default_rng(0).choice(X.shape[1], min(96, X.shape[1]), replace=False)
    for j, (tag, c) in enumerate([('typical',  cols[np.argsort(ku)[len(ku)//2]]),
                                  ('heaviest', cols[np.argmax(ku)])]):
        x = X[:, c]
        x = (x - x.mean()) / (x.std() + 1e-8)
        g = np.linspace(-6, 6, 200)
        ax[i, 2*j].hist(x, bins=120, density=True, color='steelblue')
        ax[i, 2*j].plot(g, stats.norm.pdf(g), 'r-', lw=1)
        ax[i, 2*j].set_yscale('log'); ax[i, 2*j].set_xlim(-8, 8)
        ax[i, 2*j].set_title(f'{name_of[m].split(".")[0]} ch{c} ({tag})', fontsize=8)
        stats.probplot(x, plot=ax[i, 2*j+1]); ax[i, 2*j+1].set_title('QQ', fontsize=8)
plt.tight_layout(); plt.show()

## 2. Joint shape

Gaussian marginals do not imply a Gaussian joint, and the determinant formula uses the joint.

* `proj kurt` — a vector is jointly Gaussian only if *every* 1-D projection of it is Gaussian
  (Cramer-Wold). So whiten a block of channels and test random projections of it. Projection
  kurtosis clearly above `coord kurt` means the failure is genuinely joint — the channels have
  dependent tails — and not just per-channel shape.
* `Mardia` — multivariate kurtosis divided by its Gaussian value, so 1.0 = Gaussian. Above 1 means
  rare rows where many channels spike together, and those rows dominate the covariance.

In [ ]:
def whiten(X, n_ch=32, n_rows=4000, seed=0):
    r = np.random.default_rng(seed)
    cols = r.choice(X.shape[1], min(n_ch, X.shape[1]), replace=False)
    rows = r.choice(X.shape[0], min(n_rows, X.shape[0]), replace=False)
    Z = X[np.ix_(rows, cols)].astype(np.float64)
    Z -= Z.mean(0)
    S = np.cov(Z, rowvar=False) + 1e-6*np.eye(Z.shape[1])
    w, V = np.linalg.eigh(S)
    return Z @ (V / np.sqrt(np.maximum(w, 1e-12)) @ V.T)

def joint_stats(X, n_proj=300, seed=0):
    Z = whiten(X, seed=seed)
    d = Z.shape[1]
    W = np.random.default_rng(seed).normal(size=(d, n_proj))
    W /= np.linalg.norm(W, axis=0, keepdims=True)
    pk = abs(stats.kurtosis(Z @ W, 0))
    mardia = np.mean(np.sum(Z*Z, 1)**2) / (d*(d+2))
    return np.median(abs(stats.kurtosis(Z, 0))), np.median(pk), np.percentile(pk, 95), mardia

hdr = f"{'layer':<38}{'samples':<10}{'coord kurt':>12}{'proj kurt':>11}{'proj p95':>10}{'Mardia':>9}"
print(hdr); print('-'*len(hdr))
for m in PICKS:
    for kind, X in (('per-loc', loc(m)), ('pooled', img(m))):
        ck, pk, p95, mardia = joint_stats(X)
        print(f'{name_of[m][-37:]:<38}{kind:<10}{ck:>12.2f}{pk:>11.2f}{p95:>10.2f}{mardia:>9.2f}')

## 3. Gaussian MI vs KSG MI

For a Gaussian, MI is a function of correlation alone, so the estimator only sees *linear*
dependence. A channel that drives the next layer through its magnitude (SiLU / GroupNorm gating), or
that acts symmetrically about zero, scores near 0 however much the network relies on it. That is a
live concern here: `X` is a conv output and the target is the next conv's output, so a SiLU sits
between them.

Each channel is compared against the top principal component of the next conv's activations:

* `gauss MI` — `-0.5*log(1 - rho^2)`, the quantity the estimator is built on.
* `KSG MI` — Kraskov kNN estimate, which assumes no particular distribution.
* `nonlin gain` — extra variance an `[x, x^2, |x|]` fit explains over a linear fit. The `|x|` term is
  what catches gating-style dependence.

`spearman` between the two MI columns is the number to read. Near 1.0 means Gaussianity costs
nothing where it counts. Low means channels near the pruning cut are ordered by something that
cannot see how they actually matter.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

def target_pc(m_next, rows):
    """Top principal component of the target layer's channels at the same sampled locations."""
    Y = loc(m_next)[rows].astype(np.float64)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-8)
    _, _, Vt = np.linalg.svd(Y - Y.mean(0), full_matrices=False)
    return Y @ Vt[0]

def lin_vs_nonlin(x, y):
    """R^2 of a linear fit, and of a fit that can also use x^2 and |x|."""
    z = lambda a: (a - a.mean()) / (a.std() + 1e-8)
    x, y = z(x), z(y)
    def r2(A):
        A = np.column_stack([A, np.ones(len(A))])
        res = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
        return 1 - res.var()/y.var()
    return r2(x[:, None]), r2(np.column_stack([x, x**2, abs(x)]))

def linearity_report(m, m_next, n_ch=48, n_rows=4000, seed=0):
    r = np.random.default_rng(seed)
    X = loc(m)
    rows = r.choice(X.shape[0], min(n_rows, X.shape[0]), replace=False)
    cols = r.choice(X.shape[1], min(n_ch, X.shape[1]), replace=False)
    y = target_pc(m_next, rows)
    out = []
    for c in cols:
        x = X[rows, c].astype(np.float64)
        if x.std() < 1e-6:
            continue
        rho = np.corrcoef(x, y)[0, 1]
        mi_gauss = -0.5*np.log(max(1 - rho**2, 1e-12))
        mi_ksg = float(mutual_info_regression(x[:, None], y, n_neighbors=3, random_state=0)[0])
        r2_lin, r2_nl = lin_vs_nonlin(x, y)
        out.append((mi_gauss, mi_ksg, r2_lin, r2_nl - r2_lin))
    return np.array(out)

pairs = [(convs[i], convs[i+1]) for i in [imp._order[m] for m in PICKS] if i+1 < len(convs)]
fig, ax = plt.subplots(1, len(pairs), figsize=(3.3*len(pairs), 3.2), squeeze=False); ax = ax[0]
hdr = f"{'layer':<38}{'spearman':>10}{'nonlin gain':>13}{'frac nonlin>lin':>17}"
print(hdr); print('-'*len(hdr))
for k, (m, m_next) in enumerate(pairs):
    mi_gauss, mi_ksg, r2_lin, nl_gain = linearity_report(m, m_next).T
    sp = stats.spearmanr(mi_gauss, mi_ksg).statistic
    print(f'{name_of[m][-37:]:<38}{sp:>10.3f}{np.median(nl_gain):>13.4f}'
          f'{np.mean(nl_gain > r2_lin):>17.2f}')
    ax[k].scatter(mi_gauss, mi_ksg, s=12, c=nl_gain, cmap='viridis')
    ax[k].set_xlabel('Gaussian MI'); ax[k].set_ylabel('KSG MI' if k == 0 else '')
    ax[k].set_title(f'spearman = {sp:.2f}', fontsize=9)
plt.tight_layout(); plt.show()

## 4. Conditioning on `t`

A Gaussian joint implies `Cov(X | cond)` does not depend on `cond`. The estimator conditions on `t`
by putting 9 Fourier features of `t` in the covariance, which removes a linear trend in the *mean*
and nothing else. But activation *scale* swings hard with `t`, and that has a specific cost: two
channels that are independent given `t` still look correlated, because they are loud at the same
timesteps. The criterion conditions on the other channels, so this fake correlation reads as
redundancy and pushes conditional MI down.

* `sd ratio` — largest / smallest per-channel std across `t` bins. 1.0 means scale is constant.
* `|r| linear` — mean |correlation| after the conditioning the estimator actually applies.
* `|r| binned` — the same, computed inside `t` bins, which removes mean *and* variance effects.
* `gap` = `|r| linear - |r| binned`. Positive means fake correlation the estimator leaves behind.

The `pos` columns repeat all of this for spatial position, which `_loc_embedding` conditions on the
same mean-only way.

Read `gap`, not the `|r|` columns on their own. `|r|` is inflated at small row counts and `noise`
shows how much of it is that inflation; both `|r|` columns use equal-sized row subsets so the
inflation cancels in the gap. A real effect is positive at every depth.

In [ ]:
def binned(v, nbins=10):
    e = np.quantile(v, np.linspace(0, 1, nbins+1))
    return np.clip(np.digitize(v, e[1:-1]), 0, nbins-1)

def resid(X, cond):
    A = np.column_stack([cond, np.ones(len(cond))])
    return X - A @ np.linalg.lstsq(A, X, rcond=None)[0]

def mean_abs_offdiag(X):
    R = np.corrcoef(X, rowvar=False)
    return np.nanmean(abs(R[np.triu_indices(len(R), 1)]))

def mean_abs_offdiag_matched(X, n, reps, seed=0):
    """Same as above but on random n-row subsets, so it carries the same small-sample
    inflation as a per-bin estimate computed from n rows."""
    r = np.random.default_rng(seed)
    return np.mean([mean_abs_offdiag(X[r.choice(len(X), n, replace=False)]) for _ in range(reps)])

def mixing_report(m, cond, group, n_ch=48, seed=0):
    r = np.random.default_rng(seed)
    X = loc(m)
    X = X[:, r.choice(X.shape[1], min(n_ch, X.shape[1]), replace=False)].astype(np.float64)
    ks = np.arange(group.max() + 1)
    sizes = np.array([np.sum(group == k) for k in ks])
    sd = np.stack([X[group == k].std(0) for k in ks])
    r_bin = np.average([mean_abs_offdiag(X[group == k]) for k in ks], weights=sizes)
    r_lin = mean_abs_offdiag_matched(resid(X, cond), int(sizes.mean()), len(ks))
    noise = np.sqrt(2/(np.pi*sizes.mean()))          # E|r| under independence at this subset size
    return np.median(sd.max(0) / (sd.min(0) + 1e-12)), r_lin, r_bin, noise, sd

bins_t = binned(t_loc)
bins_pos = binned(coords[:, 0], 3) * 3 + binned(coords[:, 1], 3)      # 3x3 spatial cells
hdr = (f"{'layer':<30}{'t sd ratio':>11}{'|r| linear':>12}{'|r| binned':>12}{'gap':>8}{'noise':>8}"
       f"{'pos sd ratio':>14}{'|r| linear':>12}{'|r| binned':>12}{'gap':>8}{'noise':>8}")
print(hdr); print('-'*len(hdr))
sd_by_layer = {}
for m in PICKS:
    sd_t, lin_t, bin_t, noise_t, sd = mixing_report(m, COND_T, bins_t)
    sd_p, lin_p, bin_p, noise_p, _ = mixing_report(m, COND_P, bins_pos)
    sd_by_layer[m] = sd
    print(f'{name_of[m][-29:]:<30}{sd_t:>11.1f}{lin_t:>12.3f}{bin_t:>12.3f}'
          f'{lin_t-bin_t:>8.3f}{noise_t:>8.3f}'
          f'{sd_p:>14.1f}{lin_p:>12.3f}{bin_p:>12.3f}{lin_p-bin_p:>8.3f}{noise_p:>8.3f}')

fig, ax = plt.subplots(1, len(sd_by_layer), figsize=(3.2*len(sd_by_layer), 3), squeeze=False); ax = ax[0]
for k, (m, sd) in enumerate(sd_by_layer.items()):
    ax[k].plot(np.quantile(t_loc, np.linspace(0.05, 0.95, sd.shape[0])), sd[:, :16], alpha=.7)
    ax[k].set_yscale('log'); ax[k].set_xlabel('timestep t')
    ax[k].set_ylabel('per-channel std' if k == 0 else '')
    ax[k].set_title(name_of[m].split('.')[0], fontsize=8)
plt.tight_layout(); plt.show()

## 5. Stability

Set Gaussianity aside for a moment: the formula inverts a covariance estimated from finite,
correlated samples. The `LOCS` rows drawn from one image are not independent, so the effective sample
size is well below `n_images * LOCS`, and the ridge that keeps the inverse well-behaved also biases
the MI.

* `shrink 1e-3` / `shrink 1e-1` — ranking agreement with the `1e-2` default. Near 1.0 means the ridge
  is not driving the answer.
* `split-half` — agreement between rankings computed from two disjoint halves of the data. If this is
  low, sample size is the binding constraint, not Gaussianity.
* `frac<=0` — channels whose CMI comes out non-positive. `_greedy` clamps these to 0, so they tie and
  `argmin` breaks the tie by channel index.

In [ ]:
def gauss_cmi(X, cond, target, shrink=1e-2):
    """Per-channel Gaussian CMI I(x_c ; target | other channels, cond). Same closed form as
    mi_importance.py for scalar blocks, where the determinant ratio reduces to a ratio of
    precision diagonals. On numpy so we can slice rows for the split-half test."""
    z = lambda A: (A - A.mean(0)) / (A.std(0) + 1e-8)
    X, cond, target = z(X.astype(np.float64)), z(cond.astype(np.float64)), z(target.astype(np.float64))
    C = X.shape[1]
    def prec_diag(M):
        S = np.cov(M, rowvar=False)
        S = S + shrink * np.trace(S)/len(S) * np.eye(len(S))
        return np.diag(np.linalg.inv(S))[:C], np.linalg.cond(S)
    base, cond_num = prec_diag(np.hstack([X, cond]))
    full, _ = prec_diag(np.hstack([X, cond, target]))
    return 0.5*np.log(full/base), cond_num

def stability(m, m_next, n_ch=64, n_tgt=64, seed=0):
    r = np.random.default_rng(seed)
    X, Y = loc(m), loc(m_next)
    X = X[:, r.choice(X.shape[1], min(n_ch, X.shape[1]), replace=False)]
    Y = Y[:, r.choice(Y.shape[1], min(n_tgt, Y.shape[1]), replace=False)]
    base, cond_num = gauss_cmi(X, COND, Y)
    agree = {s: stats.spearmanr(base, gauss_cmi(X, COND, Y, shrink=s)[0]).statistic
             for s in (1e-3, 1e-1)}
    h = np.arange(len(X)) < len(X)//2                # rows are image-major, so this splits by image
    first = gauss_cmi(X[h], COND[h], Y[h])[0]
    second = gauss_cmi(X[~h], COND[~h], Y[~h])[0]
    split = stats.spearmanr(first, second).statistic
    return X.shape, cond_num, agree, split, (base <= 0).mean()

hdr = (f"{'layer':<34}{'rows':>7}{'dim':>6}{'cond':>10}"
       f"{'shrink 1e-3':>13}{'shrink 1e-1':>13}{'split-half':>12}{'frac<=0':>9}")
print(hdr); print('-'*len(hdr))
for m, m_next in pairs:
    shape, cond_num, agree, split, frac0 = stability(m, m_next)
    print(f'{name_of[m][-33:]:<34}{shape[0]:>7}{shape[1]:>6}{cond_num:>10.1e}'
          f'{agree[1e-3]:>13.3f}{agree[1e-1]:>13.3f}{split:>12.3f}{frac0:>9.2f}')

## What to do about it

| finding | fix |
|---|---|
| Test 3 `spearman` high, test 4 `gap` small | Nothing. Gaussianity is fine for ranking. |
| Test 4 `gap` large | Z-score each channel *within* a `t` bin before building the covariance, or fit per `t` bucket and average the CMI. |
| Test 3 `spearman` low, `nonlin gain` large | Add `\|x\|` as a second column per channel and score it as a block of size 2 — the grouped form already supports this via `block=b`. Or add a KSG term. |
| Test 5 `split-half` low | Raise `--mi_num_batches` first; nothing else will show through the noise. |